# Devoir 3 - Implémentation et comparaison des architectures RAG

Sujet personnel: Génération de règles Snort à partir de descriptions en langage naturel.

Ce notebook suit le TP: baseline sans RAG, RAG classique, re-ranking, hybride, multi-hop, graph RAG et agentic RAG. Les APIs OpenAI/Mistral/Ollama sont interdites; la génération est donc locale et transparente.

In [ ]:
import sys
from pathlib import Path
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT / "src"))
PROJECT_ROOT

## 1. Préparation du dataset

Le notebook ne régénère pas le dataset. Il charge uniquement le dataset officiel Person 1 dans `data/processed/final_snort_dataset.csv` afin d'éviter l'écrasement des artefacts de validation.

In [ ]:
import pandas as pd
df = pd.read_csv(PROJECT_ROOT / "data" / "processed" / "final_snort_dataset.csv")
df.head(3)

In [ ]:
df["attack_type"].value_counts()

## 2. Initialisation des architectures RAG

In [ ]:
from snort_rag.architectures import SnortRAGArchitectures
rag = SnortRAGArchitectures(PROJECT_ROOT / "data" / "processed" / "final_snort_dataset.csv")

## 3. Test sur une requête Snort

In [ ]:
query = "Detect SQL injection with UNION SELECT in HTTP URI"
results = rag.run_all(query)
for name, res in results.items():
    print("=", name)
    print("attack_type:", res["attack_type"])
    print("rule:", res["generated_rule"])
    print("retrieved:", res["retrieved_ids"][:3])
    print()

## 4. Comparaison avec les métriques du Devoir 3

Métriques utilisées: accuracy de classification du type d'attaque, precision/recall/F1 macro, validité syntaxique des règles, retrieval hit@3, couverture des options Snort, risque d'hallucination.

In [ ]:
from snort_rag.evaluation import evaluate_architectures, plot_tsne
summary = evaluate_architectures(rag, PROJECT_ROOT / "results")
summary

## 5. Visualisation t-SNE des embeddings

In [ ]:
plot_tsne(rag, PROJECT_ROOT / "results" / "embedding_tsne.png")
from IPython.display import Image
Image(filename=str(PROJECT_ROOT / "results" / "embedding_tsne.png"))

## 6. Analyse critique finale

Dans ces tests, les architectures RAG atteignent une meilleure robustesse que la baseline parce qu'elles récupèrent des exemples similaires et justifient la génération. L'agentic RAG est le plus adapté au projet, car il choisit la stratégie selon l'ambiguïté de la requête et réduit le risque d'hallucination. La baseline peut produire une règle valide, mais elle ne cite aucun contexte récupéré et a donc le risque d'hallucination le plus élevé.